In [2]:
import os
import pandas as pd

# Path to local directory (adjust as needed)
directory = "C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified"

# Step 1: Collect summary files
summary_files = [f for f in os.listdir(directory) if f.startswith("summary_lambda") and f.endswith(".csv")]

# Step 2: Read all summaries with lambda/alpha extraction
records = []
for filename in summary_files:
    filepath = os.path.join(directory, filename)
    df = pd.read_csv(filepath)
    lam = float(filename.split("lambda")[1].split("_")[0])
    alpha = float(filename.split("alpha")[1].replace(".csv", ""))
    row = df.iloc[0].copy()
    row["lambda"] = lam
    row["alpha"] = alpha
    records.append(row)

summary_df = pd.DataFrame(records)

# Step 3: Pivot into 2D tables (lambda x alpha) for each metric
metrics = [
    "coverage_support", "coverage_null", "ci_width_support",
    "mean_abs_bias", "precision", "recall", "fdr", "jaccard", "snr"
]

pivot_tables = {
    metric: summary_df.pivot(index="lambda", columns="alpha", values=metric)
    for metric in metrics
}

# Step 4: Export each table to CSV
output_dir = os.path.join(directory, "tables")
os.makedirs(output_dir, exist_ok=True)

for metric, table in pivot_tables.items():
    csv_path = os.path.join(output_dir, f"{metric}_table.csv")
    table.to_csv(csv_path, float_format="%.4f")
    print(f"Saved: {csv_path}")


# display the results of the dataframes
for metric, table in pivot_tables.items():
    print(f"\n{metric} Table:")
    print(table)
    print("\n")

Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\coverage_support_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\coverage_null_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\ci_width_support_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\mean_abs_bias_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\precision_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\recall_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\fdr_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified\tables\jaccard_table.csv
Saved: C:/Users/ptboe/vscode/VU/Bootstrapping-Thesis/results/gaussian/strong/modified

In [7]:
import os
import pandas as pd
import numpy as np

# 1. Load all summaries for a given error/signal/method folder
def load_summary_tables(directory):
    summary_files = [f for f in os.listdir(directory) if f.startswith("summary_lambda") and f.endswith(".csv")]
    records = []
    for filename in summary_files:
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath)
        lam = float(filename.split("lambda")[1].split("_")[0])
        alpha = float(filename.split("alpha")[1].replace(".csv", ""))
        row = df.iloc[0].copy()
        row["lambda"] = lam
        row["alpha"] = alpha
        records.append(row)
    return pd.DataFrame(records)


# 2. Compute tail asymmetry per tracked file
def compute_tail_asymmetry(tracked_dir, j_indices):
    tail_data = []
    for j in j_indices:
        tracked_files = [f for f in os.listdir(tracked_dir) if f.startswith(f"tracked_j{j}_")]
        for f in tracked_files:
            path = os.path.join(tracked_dir, f)
            df = pd.read_csv(path)
            lam = float(f.split("lambda")[1].split("_")[0])
            alpha = float(f.split("alpha")[1].replace(".csv", ""))
            mean_asym = df["tail_asym"].mean()
            tail_data.append({"lambda": lam, "alpha": alpha, f"tail_asym_j{j}": mean_asym})
    return pd.DataFrame(tail_data)


# 3. Compute pivot tables (lambda x alpha)
def generate_metric_pivots(summary_df, tracked_df=None, metrics=None):
    if metrics is None:
        metrics = [
            "coverage_support", "coverage_null", "ci_width_support",
            "mean_abs_bias", "precision", "recall", "fdr", "jaccard", "snr"
        ]
    
    pivot_tables = {
        metric: summary_df.pivot(index="lambda", columns="alpha", values=metric)
        for metric in metrics
    }

    # Merge in tracked tail asymmetry if present
    if tracked_df is not None:
        tail_asym_cols = [col for col in tracked_df.columns if col.startswith("tail_asym")]
        for col in tail_asym_cols:
            pivot_tables[col] = tracked_df.pivot(index="lambda", columns="alpha", values=col)

    return pivot_tables


# 4. Generate comparison table across 4 methods
def load_summary_across_methods(base_path, error_type, signal_type, metric):
    method_dfs = []
    for method in ["modified", "wild", "naive", "block"]:
        method_dir = os.path.join(base_path, error_type, signal_type, method)
        summary_df = load_summary_tables(method_dir)
        summary_df = summary_df[["lambda", "alpha", metric]].copy()
        summary_df["method"] = method
        method_dfs.append(summary_df)

    all_df = pd.concat(method_dfs, ignore_index=True)
    pivot = all_df.pivot_table(index=["lambda", "alpha"], columns="method", values=metric)
    return pivot.reset_index()


summary_df = load_summary_tables("results/gaussian/strong/modified")
tracked_df = compute_tail_asymmetry("results/gaussian/strong/modified", [5, 22])
pivots = generate_metric_pivots(summary_df, tracked_df)

ValueError: Index contains duplicate entries, cannot reshape

In [8]:
import os
import pandas as pd

# Updated function to load and pivot metrics across 4 methods
def load_and_pivot_all_methods(base_path, error_type, signal_type, metric):
    records = []
    for method in ["modified", "wild", "naive", "block"]:
        method_dir = os.path.join(base_path, error_type, signal_type, method)
        files = [f for f in os.listdir(method_dir) if f.startswith("summary_lambda") and f.endswith(".csv")]
        for f in files:
            lam = float(f.split("lambda")[1].split("_")[0])
            alpha = float(f.split("alpha")[1].replace(".csv", ""))
            df = pd.read_csv(os.path.join(method_dir, f))
            val = df.iloc[0][metric]
            records.append({"lambda": lam, "alpha": alpha, "method": method, metric: val})
    
    full_df = pd.DataFrame(records)
    pivot_df = full_df.pivot_table(index="lambda", columns=["alpha", "method"], values=metric)
    return pivot_df.reset_index()

# Example usage
example_metric = "coverage_support"
pivot_df = load_and_pivot_all_methods("results", "gaussian", "strong", example_metric)
pivot_df.head()


alpha   lambda 0.099                      0.199                      0.398  \
method         block modified naive  wild block modified naive  wild block   
0       0.0656  0.20     0.20  0.22  0.22  0.20     0.20  0.22  0.22  0.20   
1       0.1312  0.18     0.16  0.16  0.16  0.18     0.16  0.16  0.22  0.18   
2       0.2625  0.04     0.06  0.06  0.06  0.04     0.06  0.06  0.06  0.04   

alpha                        
method modified naive  wild  
0          0.20  0.22  0.22  
1          0.16  0.16  0.22  
2          0.06  0.06  0.06

In [ ]:
def clean_grouped_pivot(base_path, error_type, signal_type, metric, alphas=[0.099, 0.199]):
    method_blocks = []
    for method in ["naive", "modified", "wild", "block"]:
        method_dir = os.path.join(base_path, error_type, signal_type, method)
        if not os.path.exists(method_dir):
            continue

        rows = []
        files = [f for f in os.listdir(method_dir) if f.startswith("summary_lambda")]
        for f in files:
            lam = float(f.split("lambda")[1].split("_")[0])
            alpha = float(f.split("alpha")[1].replace(".csv", ""))
            if round(alpha, 3) not in alphas:
                continue
            val = pd.read_csv(os.path.join(method_dir, f)).iloc[0][metric]
            rows.append({"lambda": lam, "alpha": round(alpha, 3), metric: val})
        
        df = pd.DataFrame(rows)
        pivot = df.pivot(index="lambda", columns="alpha", values=metric)
        pivot.columns = [f"{method}_α={a}" for a in pivot.columns]
        method_blocks.append(pivot)

    grouped = pd.concat(method_blocks, axis=1).sort_index()
    return grouped.reset_index()

grouped_cov = clean_grouped_pivot("results", "gaussian", "strong", "boot_var_support")

# Example usage of the grouped pivot
print(grouped_cov)


   lambda  naive_α=0.099  naive_α=0.199  modified_α=0.099  modified_α=0.199  \
0  0.0656           0.22           0.22              0.20              0.20   
1  0.1312           0.16           0.16              0.16              0.16   
2  0.2625           0.06           0.06              0.06              0.06   

   wild_α=0.099  wild_α=0.199  block_α=0.099  block_α=0.199  
0          0.22          0.22           0.20           0.20  
1          0.16          0.22           0.18           0.18  
2          0.06          0.06           0.04           0.04  


In [49]:
import os
import pandas as pd

# Final visualization structure: 4 separate 3x3 tables (lambda × alpha) for each method

def load_3x3_grids_per_method(base_path, error_type, signal_type, metric):
    method_grids = {}

    for method in ["naive", "modified", "wild", "block"]:
        method_dir = os.path.join(base_path, error_type, signal_type, method)
        if not os.path.exists(method_dir):
            continue

        records = []
        summary_files = [f for f in os.listdir(method_dir) if f.startswith("summary_lambda") and f.endswith(".csv")]
        for f in summary_files:
            lam = float(f.split("lambda")[1].split("_")[0])
            alpha = float(f.split("alpha")[1].replace(".csv", ""))
            val = pd.read_csv(os.path.join(method_dir, f)).iloc[0][metric]
            records.append({"lambda": lam, "alpha": alpha, metric: val})

        df = pd.DataFrame(records)
        pivot = df.pivot(index="lambda", columns="alpha", values=metric)
        method_grids[method] = pivot.sort_index(axis=0).sort_index(axis=1)

    return method_grids

        # summary = {
        #     "method": self.method,
        #     "lambda_val": self.lambda_val,
        #     "threshold_val": self.threshold_val,
        #     "signal_type": self.signal_type,
        #     "error_type": self.error_type,
        #     "coverage_support": avg("coverage_rate_support"),
        #     "coverage_null": avg("null_coverage_rate"),
        #     "ci_width_support": avg("avg_ci_width_support"),
        #     "mean_abs_bias": avg("mean_abs_bias_support"),
        #     "precision": avg("precision"),
        #     "recall": avg("recall"),
        #     "fdr": avg("fdr"),
        #     "jaccard": avg("jaccard"),
        #     "snr": avg("snr"),
        #     "boot_var_support": np.mean(boot_var_acc),
        #     "boot_bias_support": np.mean(boot_bias_acc),
        #     "tail_asym_support": np.mean(tail_asym_acc)
        # }

# Example usage:
# mean_abs_bias
# coverage_support
# ci_width_support
results = load_3x3_grids_per_method("results", "ar1", "strong", "fdr")
for method, grid in results.items():
    print(f"\n=== {method.upper()} ===")
    print(grid.round(3))



=== NAIVE ===
alpha   0.133  0.267  0.533
lambda                     
0.0675  0.873  0.873  0.873
0.1350  0.693  0.692  0.692
0.2700  0.155  0.150  0.148

=== MODIFIED ===
alpha   0.133  0.267  0.533
lambda                     
0.0675  0.873  0.873  0.873
0.1350  0.694  0.692  0.691
0.2700  0.153  0.156  0.159

=== WILD ===
alpha   0.133  0.267  0.533
lambda                     
0.0675  0.872  0.873  0.873
0.1350  0.693  0.694  0.689
0.2700  0.150  0.150  0.157

=== BLOCK ===
alpha   0.133  0.267  0.533
lambda                     
0.0675  0.872  0.872  0.873
0.1350  0.696  0.693  0.694
0.2700  0.154  0.153  0.152
